# Chapter 9 · Fine-tuning with LoRA

Fine-tune Qwen2.5-0.5B-Instruct with LoRA so that it answers in the North Cycles style (greeting, brief answer and signature), and measure before and after what it learns: the style, very well; the facts, little and badly.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*. With a GPU, the fine-tuning takes under a minute. Without a GPU, each step of 8 examples takes about 30 seconds: the notebook then uses 60 examples and one epoch (about four minutes) and does not load the large reference model.

The full explanation is in the PDF chapter. This notebook is the lab.

📘 The book on Amazon: [paperback](https://www.amazon.com/dp/B0HMX9N1KQ) · [Kindle](https://www.amazon.com/dp/B0HMX44VJ4)

## 1. Install PEFT and load

In [ ]:
!pip install -q peft

In [ ]:
"""Chapter 9: fine-tuning with LoRA. Better with a GPU; on a CPU, with less data."""
import json
import random
import re
import time
import urllib.request
from pathlib import Path

import torch
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

TUNED_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # the one we fine-tune
REFERENCE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/en/"
SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"



def load(name):
    """A model and its tokenizer, on the GPU if there is one."""
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
    return tok, model.to(DEVICE)


def download(name):
    """A JSON file from the data folder; on Colab, it is downloaded."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
WITH_GPU = DEVICE == "cuda"
VARIANTS, EPOCHS = (3, 3) if WITH_GPU else (1, 1)
print("GPU:", WITH_GPU, "· variants per question:", VARIANTS, "· epochs:", EPOCHS)

## 2. The training data

In [ ]:
GREETING, SIGNATURE = "Hi there!", "Best regards,\nThe North Cycles team"


def with_style(answer):
    """The answer in the store's style: greeting, answer and signature."""
    return f"{GREETING} {answer}\n\n{SIGNATURE}"


def training_examples(variants=3):
    """Question-answer pairs built from the FAQ: each question (the original one
    and the chapter 4 query) is written in several ways, and its answer is in
    the store's style."""
    faq = download("faq.json")
    answers = {q["id"]: q["answer"] for q in faq["questions"]}
    questions = ([(q["question"], q["id"]) for q in faq["questions"]]
                 + [(text, i) for text, i in faq["queries"]])
    before = ["", "Hi, ", "Good afternoon. ", "Quick question: ", "Good morning, "]
    after = ["", " Thanks.", " Cheers.", " Thank you very much."]
    rng = random.Random(SEED)
    pairs = []
    for question, ident in questions:
        for _ in range(variants):
            text = rng.choice(before) + question + rng.choice(after)
            pairs.append((text, with_style(answers[ident])))
    rng.shuffle(pairs)
    return pairs

In [ ]:
pairs = training_examples(VARIANTS)
print(len(pairs), "examples. Two of them:\n")
for question, reply in pairs[:2]:
    print("Customer:", question)
    print("Answer:", reply, "\n")

## 3. Only the answer counts

In [ ]:
def encode(tok, question, answer):
    """Tokens of the whole conversation and labels for the loss: -100 on the
    question part, so that only the answer is learned."""
    messages = [{"role": "user", "content": question}]
    prompt = tok.apply_chat_template(messages, tokenize=False,
                                     add_generation_prompt=True)
    full = prompt + answer + "<|im_end|>"
    ids = tok(full, add_special_tokens=False)["input_ids"]
    n_prompt = len(tok(prompt, add_special_tokens=False)["input_ids"])
    labels = [-100] * n_prompt + ids[n_prompt:]
    return ids, labels


def batches(tok, pairs, size=8):
    """Groups of examples padded to the same length."""
    for i in range(0, len(pairs), size):
        encoded = [encode(tok, q, a) for q, a in pairs[i:i + size]]
        length = max(len(ids) for ids, _ in encoded)
        inputs, labels, mask = [], [], []
        for ids, lab in encoded:
            padding = length - len(ids)
            inputs.append(ids + [tok.pad_token_id] * padding)
            labels.append(lab + [-100] * padding)
            mask.append([1] * len(ids) + [0] * padding)
        yield (torch.tensor(inputs, device=DEVICE),
               torch.tensor(labels, device=DEVICE),
               torch.tensor(mask, device=DEVICE))

In [ ]:
tok, model = load(TUNED_MODEL)
ids, labels = encode(tok, *pairs[0])
print(len(ids), "tokens, of which count in the loss", sum(e != -100 for e in labels))
print("Part that is learned:", repr(tok.decode([t for t, e in zip(ids, labels) if e != -100])))

## 4. Test before fine-tuning

In [ ]:
def answer(tok, model, question, system=None, max_tokens=120):
    """The model's answer to a question, with temperature 0."""
    messages = ([{"role": "system", "content": system}] if system else []) + [
        {"role": "user", "content": question}]
    text = tok.apply_chat_template(messages, tokenize=False,
                                   add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    return tok.decode(output[0, inputs["input_ids"].shape[1]:],
                      skip_special_tokens=True).strip()


STYLE_INSTRUCTION = (
    "You are the assistant of North Cycles, a bicycle store. Answer in English, "
    "in a friendly, informal tone, in one or two sentences. Always start with "
    "\"Hi there!\" and always end with the signature \"Best regards,\" and, on "
    "the next line, \"The North Cycles team\".")


# Ten questions about facts that appear in the training data, in other words,
# with the fact the answer must contain; and ten about things the training
# data does not cover.
KNOWN = [
    ("How many days does a regular delivery to Chicago take?", "2-3"),
    ("What do you charge for shipping if my order is $20?", "4.95"),
    ("Do you send packages to Maui?", "5-7"),
    ("How long do I have to return something I don't like?", "30 days"),
    ("How long does it take you to give me my money back?", "14 days"),
    ("How many years of warranty does an e-bike have?", "2-year"),
    ("Can I pay with Apple Pay?", "Apple Pay"),
    ("What's the minimum amount to finance a purchase?", "300"),
    ("How much are the gift cards?", "25"),
    ("How long does the battery take to charge fully?", "4 and 6"),
]


NEW = [
    "How much is the Aero 3?", "Do you have carbon road bikes?",
    "Where is my order NC-10871?", "Do you organize group rides?",
    "Do you sell electric scooters?",
    "What tire pressure should a mountain bike have?",
    "Do you have student discounts?", "Can I park near the store?",
    "Which helmet do you recommend for commuting to work?",
    "How much does the Folding 20 weigh?"]


def check_style(text):
    """Does it start with the greeting, end with the signature and is it brief?"""
    return {"greeting": text.startswith(GREETING),
            "signature": text.rstrip(".").endswith("The North Cycles team"),
            "brief": len(text.split()) <= 70}


def evaluate(tok, model, system=None):
    """Style on the twenty questions and correct facts on the ten known ones."""
    rows = []
    for question, fact in KNOWN + [(q, None) for q in NEW]:
        a = answer(tok, model, question, system)
        rows.append({"question": question, "answer": a, **check_style(a),
                     "fact": None if fact is None else fact in a})
    return rows


def summary(rows):
    """How many answers meet each criterion."""
    return {"greeting": sum(r["greeting"] for r in rows),
            "signature": sum(r["signature"] for r in rows),
            "brief": sum(r["brief"] for r in rows),
            "facts": sum(bool(r["fact"]) for r in rows)}

In [ ]:
results = {}
results["0.5B without instruction"] = evaluate(tok, model)
results["0.5B with instruction"] = evaluate(tok, model, STYLE_INSTRUCTION)
for name, rows in results.items():
    print(f"{name:26s}", summary(rows))

## 5. Fine-tune with LoRA

In [ ]:
def fine_tune(tok, model, pairs, epochs=3, rate=2e-4, rank=8):
    """Adds LoRA matrices of rank `rank` to the linear layers of the attention
    and the MLP, and trains only those matrices."""
    config = LoraConfig(r=rank, lora_alpha=2 * rank, lora_dropout=0.05,
                        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                        "gate_proj", "up_proj", "down_proj"],
                        task_type="CAUSAL_LM")
    model = get_peft_model(model, config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:,} of {total:,} "
          f"({100 * trainable / total:.2f}%)")
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=rate)
    torch.manual_seed(SEED)
    model.train()
    losses, start = [], time.time()
    for epoch in range(epochs):
        for inputs, labels, mask in batches(tok, pairs):
            loss = model(input_ids=inputs, attention_mask=mask,
                         labels=labels).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            losses.append(loss.item())
        print(f"Epoch {epoch + 1}: mean loss "
              f"{sum(losses[-len(pairs) // 8:]) / (len(pairs) // 8):.3f}")
    model.eval()
    return model, {"losses": losses, "seconds": time.time() - start,
                   "trainable": trainable, "total": total}

In [ ]:
model, training = fine_tune(tok, model, pairs, EPOCHS)
print(f"Training: {training['seconds']:.0f} s")
results["0.5B fine-tuned with LoRA"] = evaluate(tok, model)
print(summary(results["0.5B fine-tuned with LoRA"]))

In [ ]:
import matplotlib.pyplot as plt
plt.plot(training["losses"])
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.show()

The answers of the fine-tuned model. Look at the style and at the facts, especially on the new questions:

In [ ]:
for r in results["0.5B fine-tuned with LoRA"]:
    print("•", r["question"])
    print("  ", r["answer"].replace("\n", " / "))

## 6. Save and load the adapter

In [ ]:
def load_fine_tuned(folder="north_cycles_adapter"):
    """The base model with the saved LoRA adapter."""
    tok, base = load(TUNED_MODEL)
    return tok, PeftModel.from_pretrained(base, folder).eval()

In [ ]:
model.save_pretrained("north_cycles_adapter")
import os
print({f: f"{os.path.getsize(os.path.join('north_cycles_adapter', f)) / 1e6:.1f} MB"
       for f in os.listdir("north_cycles_adapter")})

## 7. Compare with the large model (GPU only)

In [ ]:
if WITH_GPU:
    del model
    torch.cuda.empty_cache()
    tok2, reference = load(REFERENCE_MODEL)
    results["1.5B with instruction"] = evaluate(tok2, reference, STYLE_INSTRUCTION)
for name, rows in results.items():
    print(f"{name:26s}", summary(rows))

## Exercises

1. Explain in your own words what a LoRA adapter is and why it takes up so little space. What is the advantage of storing it separately from the model?

2. Work out how many parameters LoRA adds with r = 8 to an 896 × 4864 matrix, like those of the model's MLP. What fraction of the matrix is that?

3. Run the fine-tuning without a GPU with `variants=1` (60 examples) and one epoch. How long does it take? Does it learn the style?

4. Repeat the fine-tuning with 1 and with 2 epochs. Does it learn the style? How many facts does it get right? Is there a point where it learns the style without memorizing so much?

5. Try ranks of 2 and 32. Does the result change? And the number of trainable parameters?

6. Write ten training examples in a different style, for example a more formal one that addresses the customer as "Dear customer", and fine-tune the model with them alone. Are ten examples enough to change the style?

7. Add to the training data examples of questions the store cannot answer, with the answer "I don't have that information; I'm passing your question to the team", in the store's style. Does the fine-tuned model make things up less on the new questions?

8. Use the fine-tuned model with a system message that contains the full FAQ. Does it keep the style? Does it get more facts right? Compare it with the base model given the same information and the style instruction.

9. Load the saved adapter with `load_fine_tuned` in a new session and check that it answers the same way. Then merge the adapter into the model with `merge_and_unload` and compare the response speed.

10. **Fine-tuning model card.** Write a short card for the adapter: training data, configuration, time, evaluation results and known limitations, like the one that should accompany any model that goes into production.

## References

Models [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) and [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct). LoRA: [Hu et al. (2021)](https://arxiv.org/abs/2106.09685). QLoRA: [Dettmers et al. (2023)](https://arxiv.org/abs/2305.14314). [PEFT](https://huggingface.co/docs/peft). The North Cycles store and its data are fictional. Our own code and experiments.